<a href="https://colab.research.google.com/github/boeun0905/boeun-study/blob/main/%ED%9A%8C%EC%9D%98%EB%A1%9D%EC%9A%94%EC%95%BD%EB%B0%8FPPT%EC%83%9D%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyPDF2 python-docx
!pip install python-pptx
!pip install openai requests python-pptx
!pip install -U langchain-openai
!pip install -U langchain-google-genai
!pip install -U langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.43.0
    Uninstalling google-auth-2.43.0:
      Successfully uninstalled google-auth-2.43.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.55.0
    Uninstallin

In [ ]:
import os
import re
import requests
import gradio as gr
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.dml.color import RGBColor
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
import PyPDF2
from docx import Document
from openai import OpenAI

# 1. 환경 변수 로드 및 클라이언트 설정
load_dotenv(dotenv_path="/content/.env", override=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 2. 파일 텍스트 추출 함수
def extract_text(file):
    if file is None: return ""
    try:
        if file.name.endswith('.pdf'):
            with open(file.name, 'rb') as f:
                pdf_reader = PyPDF2.PdfReader(f)
                return "".join([page.extract_text() for page in pdf_reader.pages])
        elif file.name.endswith('.docx'):
            doc = Document(file.name)
            return "\n".join([para.text for para in doc.paragraphs])
    except Exception as e:
        return f"파일 추출 중 오류 발생: {str(e)}"
    return ""

# 3. DALL-E 이미지 생성 함수
def generate_slide_image(prompt_text):
    try:
        refined_prompt = f"Professional corporate 3D render or high-quality photography, no text, minimalist, modern business style, blue and white tone: {prompt_text}"
        response = client.images.generate(
            model="dall-e-3",
            prompt=refined_prompt,
            size="1024x1024", n=1
        )
        img_url = response.data[0].url
        img_path = f"temp_{hash(prompt_text)}.png"
        with open(img_path, 'wb') as f:
            f.write(requests.get(img_url).content)
        return img_path
    except Exception as e:
        print(f"이미지 생성 실패: {e}")
        return None

# 4. 메인 프로세스
def process_consulting(text_input, file_input, model_choice):
    try:
        content = extract_text(file_input) if file_input else text_input
        if not content or len(content.strip()) < 10:
            return "내용을 입력하거나 파일을 업로드해주세요.", None, None

        llm = init_chat_model(model_choice, temperature=0.2)
        sys_msg = """당신은 전략기획팀의 유능한 대리입니다. 제공되는 회의록을 바탕으로 임원 보고용 PPT 초안(10장)을 작성하세요.
                    반드시 각 슬라이드 끝에 '[이미지 추천: 묘사]'를 포함하세요.
                    **굵게 표시**를 적절히 활용하여 핵심 키워드를 강조하세요.
                    PPT의 구성은 아래와 같이 구성되어야 합니다.
                    형식: 반드시 '슬라이드 n: 제목'으로 시작할 것.
                    각 슬라이드의 상단 헤더바는 PPT내용의 요약이여야 할 것.
                   - 슬라이드 1: 제목 (회의 제목, 날짜, 작성자)
                   - 슬라이드 2: 목차 및 회의 개요
                   - 슬라이드 3~8: 핵심 논의사항 및 상세 전략 (중요도 순)
                   - 슬라이드 9: 기대 효과 및 리스크 관리
                   - 슬라이드 10: 향후 일정 및 Q&A"""

        ai_response = llm.invoke([SystemMessage(content=sys_msg), HumanMessage(content=content)]).content

        # Remove '---' from the AI response for display
        cleaned_ai_response = re.sub(r'^-+\s*', '', ai_response, flags=re.MULTILINE)
        print(cleaned_ai_response)

        prs = Presentation()
        prs.slide_width = Inches(13.333)
        prs.slide_height = Inches(7.5)

        raw_sections = re.split(r'슬라이드 \d+[:\s]*', ai_response)
        slides_content = [s.strip() for s in raw_sections if s.strip()]

        for i, section in enumerate(slides_content):
            slide = prs.slides.add_slide(prs.slide_layouts[6])

            # --- [디자인] 상단 헤더 바 추가 ---
            header_rect = slide.shapes.add_shape(1, 0, 0, prs.slide_width, Inches(0.8))
            header_rect.fill.solid()
            header_rect.fill.fore_color.rgb = RGBColor(0, 35, 102)
            header_rect.line.fill.background()

            lines = section.split('\n')
            title_text = lines[0]
            body_text = "\n".join(lines[1:])

            # --- 제목 텍스트 박스 ---
            title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.1), Inches(12), Inches(0.6))
            title_tf = title_box.text_frame
            title_p = title_tf.paragraphs[0]
            title_tf.text = title_text
            title_tf.paragraphs[0].font.bold = True
            title_tf.paragraphs[0].font.size = Pt(32)
            title_tf.paragraphs[0].font.color.rgb = RGBColor(255, 255, 255)
            title_tf.paragraphs[0].alignment = PP_ALIGN.CENTER

            img_tag = re.search(r'\[이미지 추천: (.+?)\]', body_text)
            if img_tag:
                img_path = generate_slide_image(img_tag.group(1))
                if img_path:
                    slide.shapes.add_picture(img_path, Inches(7.5), Inches(1.2), width=Inches(5.3), height=Inches(5.3))

            clean_body = re.sub(r'\[이미지 추천: .+?\]', '', body_text).strip()
            body_box = slide.shapes.add_textbox(Inches(0.5), Inches(1.2), Inches(6.5), Inches(5.8))
            tf = body_box.text_frame
            tf.word_wrap = True
            tf.vertical_anchor = MSO_ANCHOR.TOP

            for line in clean_body.split('\n'):
                if not line.strip(): continue
                p = tf.add_paragraph()
                p.space_after = Pt(10)
                p.line_spacing = 1.2

                # 불렛 포인트 처리
                content_text = line.replace('---', '').strip()
                content_text = line.replace("###", "").strip()
                # 마크다운 굵게(**) 파싱 및 색상 적용
                parts = re.split(r'(\**.*?\**)', content_text)
                for part in parts:
                    run = p.add_run()
                    if part.startswith('**') and part.endswith('**'):
                        run.text = part.replace('**', '')
                        run.font.bold = True
                        run.font.color.rgb = RGBColor(0, 150, 200)
                    else:
                        run.text = part
                    run.font.size = Pt(18)
                    run.font.name = '맑은 고딕'

        ppt_name = "Strategy_Report.pptx"
        prs.save(ppt_name)

        return cleaned_ai_response, None, ppt_name

    except Exception as e:
        return f" 오류 발생: {str(e)}", None, None

# 5. UI 구성 (Gradio)
with gr.Blocks() as demo:
    gr.Markdown("# AI를 사용한 회의록 요약 및 PPT 생성")
    with gr.Row():
        with gr.Column(scale=1):
            with gr.Tabs():

              with gr.TabItem("텍스트 입력"):
                 txt_in = gr.Textbox(label="내용 입력", lines=10)

              with gr.TabItem("파일 업로드"):
                 file_in = gr.File(label="파일 업로드", file_types=[".pdf", ".docx"])

              model_sel = gr.Dropdown(
                  choices=["openai:gpt-4o-mini", "google_genai:gemini-1.5-flash"],
                  value="openai:gpt-4o-mini", label="AI 모델 선택"
              )

            btn = gr.Button("PPT 생성하기", variant="primary")
        with gr.Column(scale=1):
          gr.Markdown("기획안 프리뷰")
          #out_txt = gr.Markdown(elem_id="preview_area")
          out_txt = gr.Textbox(elem_id="preview_area",show_label=False,lines=13)

          gr.Markdown("PPT 다운로드")
          out_file = gr.File(elem_id="file_area")

    btn.click(fn=process_consulting, inputs=[txt_in, file_in, model_sel], outputs=[out_txt, gr.State(), out_file])

if __name__ == "__main__":
    demo.launch(theme=gr.themes.Soft())

/tmp/ipython-input-1255870455.py:152: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1aac6688d3964351ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
